In [ ]:
import torch
import torch.nn as nn

from torchvision import models
from torchvision import datasets
from torchvision import transforms

from torch.utils.data import DataLoader

import pandas as pd
import numpy as np

In [ ]:
# load test dataset
test_transform = transforms.Compose([

    transforms.Resize(
        (224,224)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )

])

In [ ]:
# load folder test
test_dataset = datasets.ImageFolder(
    r"C:\Users\User\Documents\SEMESTER 7\TUGAS AKHIR\Facial-Emotion-ASD-Copy\data\resized\test",
    transform=test_transform
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

print(
    test_dataset.classes
)

In [ ]:
# testing MobileNetV2
mobilenet = models.mobilenet_v2(
    weights=None
)

num_features = (
    mobilenet.classifier[1]
    .in_features
)

mobilenet.classifier[1] = nn.Linear(
    num_features,
    2
)

mobilenet.load_state_dict(
    torch.load(
        "mobilenetv2_best.pth",
        map_location="cpu"
    )
)

mobilenet.eval()

In [ ]:
# PREDIKSI MobileNetV2

y_true_mobile = []
y_pred_mobile = []

with torch.no_grad():

    for images, labels in test_loader:

        outputs = mobilenet(
            images
        )

        _, preds = torch.max(
            outputs,
            1
        )

        y_true_mobile.extend(
            labels.numpy()
        )

        y_pred_mobile.extend(
            preds.numpy()
        )

In [ ]:
# Cek jumlah data

print(
    len(y_true_mobile)
)

print(
    len(y_pred_mobile)
)

In [ ]:
# Testing ResNet18

resnet = models.resnet18(
    weights=None
)

num_features = (
    resnet.fc.in_features
)

resnet.fc = nn.Linear(
    num_features,
    2
)

resnet.load_state_dict(
    torch.load(
        "resnet18_best.pth",
        map_location="cpu"
    )
)

resnet.eval()

In [ ]:
# prediksi Resnet18

y_true_resnet = []
y_pred_resnet = []

with torch.no_grad():

    for images, labels in test_loader:

        outputs = resnet(
            images
        )

        _, preds = torch.max(
            outputs,
            1
        )

        y_true_resnet.extend(
            labels.numpy()
        )

        y_pred_resnet.extend(
            preds.numpy()
        )

In [ ]:
# verifikasi

print(
    len(y_true_resnet)
)

print(
    len(y_pred_resnet)
)

In [ ]:
# pastikan label sama

print(
    np.array_equal(
        y_true_mobile,
        y_true_resnet
    )
)

In [ ]:
# hasil
len(y_true_mobile)
len(y_pred_mobile)
len(y_pred_resnet)

In [ ]:
# ubah prediksi jadi benar dan salalh

mobile_correct = (
    np.array(y_true_mobile)
    ==
    np.array(y_pred_mobile)
).astype(int)

resnet_correct = (
    np.array(y_true_resnet)
    ==
    np.array(y_pred_resnet)
).astype(int)

print(mobile_correct[:10])
print(resnet_correct[:10])

In [ ]:
!pip install scipy

In [ ]:
# jalankan paired t-test
from scipy.stats import ttest_rel

t_stat, p_value = ttest_rel(
    mobile_correct,
    resnet_correct
)

print(
    "t-statistic =",
    t_stat
)

print(
    "p-value =",
    p_value
)

In [ ]:
# hitung rata2 akurasi per sampel
print(
    "MobileNet Mean =",
    mobile_correct.mean()
)

print(
    "ResNet Mean =",
    resnet_correct.mean()
)

In [ ]:
import sys
import torch
import numpy
import sklearn
import matplotlib
import pandas
import cv2
import mediapipe

print("Python :", sys.version)
print("PyTorch :", torch.__version__)
print("NumPy :", numpy.__version__)
print("Scikit-Learn :", sklearn.__version__)
print("Matplotlib :", matplotlib.__version__)
print("Pandas :", pandas.__version__)
print("OpenCV :", cv2.__version__)
print("MediaPipe :", mediapipe.__version__)

In [ ]:
import torchvision

print("TorchVision :", torchvision.__version__)

In [ ]:
import streamlit as st

print(st.__version__)

In [ ]:
import matplotlib.pyplot as plt

# Data Valence-Arousal

emotions = {
    "Joy": (0.8, 0.6),
    "Sadness": (-0.7, -0.4)
}

fig, ax = plt.subplots(
    figsize=(6, 6)
)

# Garis sumbu

ax.axhline(
    y=0,
    linewidth=1
)

ax.axvline(
    x=0,
    linewidth=1
)

# Plot titik

for emotion, (valence, arousal) in emotions.items():

    ax.scatter(
        valence,
        arousal,
        s=200
    )

    ax.text(
        valence + 0.03,
        arousal + 0.03,
        emotion,
        fontsize=10
    )

# Pengaturan grafik

ax.set_xlim(-1, 1)
ax.set_ylim(-1, 1)

ax.set_xlabel(
    "Arousal"
)

ax.set_ylabel(
    "Valence"
)

ax.set_title(
    "Pemetaan Emosi pada Ruang Valence-Arousal"
)

ax.grid(True)

plt.tight_layout()

plt.savefig(
    "valence_arousal.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()